In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"

In [0]:
countries_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/netflix_countries")

In [0]:
countries_df.display()

In [0]:
silver_countries = (
    countries_df
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("country", trim(col("country")))
    .withColumn(
        "country",
        when(col("country") == "", None)
         .otherwise(col("country"))
    )
    .filter(col("show_id").isNotNull())
    .filter(col("country").isNotNull())
    .dropDuplicates(["show_id", "country"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)



In [0]:
silver_countries.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/netflix_countries")